In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay

# 1. Preprocessing and cleaning data

In [2]:
## import data
city_df = pd.read_csv('city_hotel.csv')
resort_df = pd.read_csv('resort_hotel.csv')

## Combine the two datasets
city_df['HotelType'] = 'city'
resort_df['HotelType'] = 'resort'

df = pd.concat([city_df, resort_df], axis = 0, ignore_index=True)
df.head()

,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate,hotel_type
0,0,6,2015,July,27,1,0,2,1,0.0,...,6,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-03,city
1,1,88,2015,July,27,1,0,4,2,0.0,...,9,NULL,0,Transient,76.5,0,1,Canceled,2015-07-01,city
2,1,65,2015,July,27,1,0,4,1,0.0,...,9,NULL,0,Transient,68.0,0,1,Canceled,2015-04-30,city
3,1,92,2015,July,27,1,2,4,2,0.0,...,9,NULL,0,Transient,76.5,0,2,Canceled,2015-06-23,city
4,1,100,2015,July,27,2,0,2,2,0.0,...,9,NULL,0,Transient,76.5,0,1,Canceled,2015-04-02,city


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   IsCanceled                   119390 non-null  int64  
 1   LeadTime                     119390 non-null  int64  
 2   ArrivalDateYear              119390 non-null  int64  
 3   ArrivalDateMonth             119390 non-null  object 
 4   ArrivalDateWeekNumber        119390 non-null  int64  
 5   ArrivalDateDayOfMonth        119390 non-null  int64  
 6   StaysInWeekendNights         119390 non-null  int64  
 7   StaysInWeekNights            119390 non-null  int64  
 8   Adults                       119390 non-null  int64  
 9   Children                     119386 non-null  float64
 10  Babies                       119390 non-null  int64  
 11  Meal                         119390 non-null  object 
 12  Country                      118902 non-null  object 
 13 

## Variables description
Sensitive information were replaced with codes.

| Feature | Data Type | Description |
| :--- | :--- | :--- |
| ADR | Numeric | Average Daily Rate |
| Adults | Integer | Number of adults |
| Agent | Categorical | ID of the travel agency that made the booking |
| ArrivalDateDayOfMonth | Integer | Day of the month of the arrival date |
| ArrivalDateMonth | Categorical | Month of arrival date with 12 categories: “January” to “December”
| ArrivalDateWeekNumber | Integer | Week number of the arrival date
| ArrivalDateYear | Integer | Year of arrival date
| Babies | Integer | Number of babies
| BookingChanges | Integer | Number of changes/amendments made to the booking from the moment the booking was entered on the PMS until the moment of check-in or cancellation
| Children | Integer | Number of children
| Company | Categorical | Code of the company/entity that made the booking or responsible for paying the booking. 
| Country | Categorical | Country of origin
| CustomerType | Categorical | Type of booking, assuming one of four categories: <br>1. Contract - when the booking has an allotment or other type of contract associated to it; <br>2. Group – when the booking is associated to a group; <br>3. Transient – when the booking is not part of a group or contract, and is not associated to other transient booking; <br>4. Transient-party – when the booking is transient, but is associated to at least other transient booking
| DaysInWaitingList | Integer | Number of days the booking was in the waiting list before it was confirmed to the customer
| DepositType | Categorical | Indication on if the customer made a deposit to guarantee the booking. This variable can assume three categories: <br>1. No Deposit – no deposit was made; <br>2. Non Refund – a deposit was made in the value of the total stay cost; <br>3. Refundable – a deposit was made with a value under the total cost of stay.
 | DistributionChannel | Categorical | Booking distribution channel. The term “TA” means “Travel Agents” and “TO” means “Tour Operators”
| HotelType | Categorical | "city" means city hotels whereas "resort" means resort hotels.
| IsRepeatedGuest | Categorical | Value indicating if the booking name was from a repeated guest (1) or not (0)
| LeadTime | Integer | Number of days that elapsed between the entering date of the booking into the PMS and the arrival date
| MarketSegment | Categorical | Market segment designation. In categories, the term “TA” means “Travel Agents” and “TO” means “Tour Operators”
| Meal | Categorical | Type of meal booked. Categories are presented in standard hospitality meal packages: <br>1. Undefined/SC – no meal package;<br>2. BB – Bed & Breakfast;<br>3. HB – Half board (breakfast and one other meal – usually dinner);<br>4. FB – Full board (breakfast, lunch and dinner)
| PreviousBookingsNotCanceled | Integer | Number of previous bookings not cancelled by the customer prior to the current booking
| PreviousCancellations | Integer | Number of previous bookings that were cancelled by the customer prior to the current booking
| RequiredCardParkingSpaces | Integer | Number of car parking spaces required by the customer
| ReservedRoomType | Categorical | Code of room type reserved. Code is presented instead of designation for anonymity reasons
| StaysInWeekendNights | Integer | Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel
| StaysInWeekNights | Integer | Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel
| TotalOfSpecialRequests | Integer | Number of special requests made by the customer (e.g. twin bed or high floor)
| AssignedRoomType<br><mark>(to be discarded)</mark> | Categorical | Code for the type of room finally assigned to the customers. Sometimes the assigned room type differs from the reserved room type due to hotel operation reasons (e.g. overbooking) or by customer request. Code is presented instead of designation for anonymity reasons. This information is irrelevant and should be dropped.
| ReservationStatus<br><mark>(to be discarded)</mark> | Categorical | Reservation last status, assuming one of three categories:<br>1. Canceled <br>2. Check-Out <br>3. No-Show. <br>This variable describes the same information as our prediction outcome, to avoid data leakage it should be dropped.
| ReservationStatusDate<br><mark>(to be discarded)</mark>  | Date | Date at which the ReservationStatus was set. As ReservationStatus should be removed, this variable is useless as well.
| IsCanceled<br>(Outcome) | Categorical | 	Value indicating if the booking was canceled (1) or not (0)

In [4]:
## Select one cancelled case to better understanding the meaning of the variables  
df[df['IsCanceled'] == 1].iloc[1]

IsCanceled                                    1
LeadTime                                     65
ArrivalDateYear                            2015
ArrivalDateMonth                           July
ArrivalDateWeekNumber                        27
ArrivalDateDayOfMonth                         1
StaysInWeekendNights                          0
StaysInWeekNights                             4
Adults                                        1
Children                                    0.0
Babies                                        0
Meal                                  BB       
Country                                     PRT
MarketSegment                         Online TA
DistributionChannel                       TA/TO
IsRepeatedGuest                               0
PreviousCancellations                         0
PreviousBookingsNotCanceled                   0
ReservedRoomType               A               
AssignedRoomType               A               
BookingChanges                          

In [ ]:
## drop inappropriate columns
df = df.drop(columns=['ReservationStatus', 'ReservationStatusDate', 'AssignedRoomType'])

## drop null values
df = df.dropna()
print(df.isna().sum()) # summary of null values 

In [ ]:
numeric_col = ['LeadTime','StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'PreviousCancellations', 'PreviousBookingsNotCanceled',
              'BookingChanges', 'DaysInWaitingList', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests']   

categorical_col = ['ArrivalDateYear', 'ArrivalDateMonth', 'ArrivalDateWeekNumber', 'ArrivalDateDayOfMonth', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'IsRepeatedGuest', 'ReservedRoomType', 'DepositType', 'Agent', 'Company', 'CustomerType', 'HotelType']

## Handling numeric variables

In [ ]:
## Spot problematic numeric columns
prob_numeric_col = []

for x in numeric_col:
    if df[x].dtype == 'O': # if any words have been input into a numeric column, the datatype will become object.  
        prob_numeric_col.append(x)

print(prob_numeric_col) # No probmatic numeric columns found at this stage

In [ ]:
## Preparations for outliers removal
df[numeric_col].describe()

## Except StaysInWeekendNights, StaysInWeekNights, all columns have suspicous max values
prob_numeric_col = ['LeadTime', 'Adults', 'Children', 'Babies', 'PreviousCancellations', 'PreviousBookingsNotCanceled',
              'BookingChanges', 'DaysInWaitingList', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests'] 
print(f'There are {len(prob_numeric_col)} suspicous numeric columns.') 

In [ ]:
## Draw stripplot to spot out outliers

def plot_multiple_stripplots(df: pd.DataFrame, col_list: list):
    """Pass the dataframe and the column list to draw swarmplots for chosen columns. The list specifies which columns to include."""

    ## Determine the row count of the subplot
    if len(col_list)%2 == 0:
        subplot_row_count = len(col_list)/2
    else:
        subplot_row_count = int(len(col_list)/2) + 1
        
    fig, axes = plt.subplots(subplot_row_count, 2, figsize=(9, 25))
    axes = axes.flatten() 

    ## Plot the graph
    for i, col in enumerate(col_list):
        sns.stripplot(data=df, y=col, ax=axes[i]) 
        
    plt.tight_layout()
    plt.show()

plot_multiple_stripplots(df, prob_numeric_col)

- From the above plot, 'LeadTime', 'Adults', 'Children', 'Babies', 'ADR' and 'RequiredCarParkingSpaces' have outliers or unreasonable values.
- It is puzzling to have 10 or more adults leaving in one hotel room. Having 8-10 children or babies in one family is also puzzling.



In [ ]:
## Drop outliers for LeadTime and ADR
df = df[(df['LeadTime'] < 650) & (df['ADR'] < 1000)]

In [ ]:
prob_numeric_col = ['Adults', 'Children', 'Babies', 'RequiredCarParkingSpaces']
for col in prob_numeric_col:
    print(f'Here are the unique values for {col}')
    sorted_unique_value = np.sort(df[col].unique())
    print(sorted_unique_value)
    print('\n')

In [ ]:
## Drop outliers for Adults, Children, Babies, RequiredCarParkingSpaces
mask = (df['Adults'] < 8) & (df['Children'] < 8) & (df['Babies'] < 8) & (df['RequiredCarParkingSpaces'] < 8) 
df = df[mask]

## Handling categorical variables

In [ ]:
categorical_col 

In [ ]:
# Spot problematic categorical columns
for col in categorical_col:
    print(f'Here are the unique values for {col}')
    print(df[col].unique())
    print('\n')

In [ ]:
## Strip the string values and fill back np.nan values 
def strip_and_fill_na(word: str) -> str | float:
    """ strip word or return np.nan if NULL is written"""
    word = word.strip()
    return word if (word != 'NULL') else np.nan


prob_categorical_col = ['Meal', 'ReservedRoomType', 'DepositType', 'Agent', 'Company']
for col in prob_categorical_col:
    df[col] = df[col].map(strip_and_fill_na)




In [ ]:
## Check whether the mapping is successful or not
print(f'Number of null values:\n{df.isna().sum()}')

for col in categorical_col:
    print(f'Here are the unique values for {col}')
    print(df[col].unique()) 
    print('\n')

## As the majority values in 'Company' column is NA, it is advisable to drop the whole column. 
df.drop(columns=['Company'], inplace=True)

## na values for agent are dropped as well
df.dropna(inplace=True)

In [ ]:
## Change the wording in 'Meal' and 'DistributionChannel' columns for better understanding
mapping = {
    'SC' : 'No meal',
    'Undefined': 'No meal'
}
    
df['Meal'] = df['Meal'].replace(mapping)
print(f'Here are the unique values for "Meal": {df['Meal'].unique()}')

df['DistributionChannel'] = df['DistributionChannel'].replace('Undefined', 'Other')
print(f'Here are the unique values for "DistributionChannel": {df['DistributionChannel'].unique()}')

## Descriptive Statistics

In [ ]:
## Overview of the cleaned data
print(df.info())

df.describe()
